# 분류를 위해 미세 튜닝하기 목차
* [Chapter 1 여러가지 미세 튜닝 방법](#chapter1)
* [Chapter 2 데이터셋 준비](#chapter2)

## Chapter 1 여러가지 미세 튜닝 방법 <a class="anchor" id="chapter1"></a>
1. 언어 모델을 미세 튜닝하는 가장 일반적인 방법은 지시 미세 튜닝(instruction fine-tuning)과 분류 미세 튜닝(classification fine-tuning)입니다.
   - 지시 미세 튜닝은 구체적인 지시 데이터를 사용해 일련의 작업에서 언어 모델을 훈련한다.
      - 모델이 사용자의 구체적인 지시를 이해하고 이를 기반으로 응답을 생성하는 능력을 향상시킨다.
      - 복잡한 사용자의 지시를 기반으로 다양한 작업을 처리해야하는 모델에 잘 맞는다.
      - 다양한 작업에 능숙한 모델을 개발하려면 데이터셋과 컴퓨팅 자원이 많이 필요하다.
      - 예) 영어 문장을 독일어로 변경하라는 지시를 수행한다
   - 분류 미세 튜닝은 레이블이 있는 데이터셋을 사용해 모델이 특정 클래스에 대한 예측을 수행하도록 훈련한다.
      - 감성 분석이나 스팸 감지와같은 데이터를 사전에 정의된 클래스로 정확히 분류해야 하는 프로젝트에 적합하다
      - 데이터와 컴퓨팅 자원이 비교적 적게 필요하지만 모델이 훈련된 특정 클래스로만 사용이 제한된다.
      - 예) 모델이 텍스크가 스펨인지 아닌지 결정하는 작업을 수행한다.

      ![메세 튜닝](image/06-01-tunning2.png)

      ![메세 튜닝2](image/06-01-tunning4.png)



## Chapter 2 데이터셋 준비 <a class="anchor" id="chapter2"></a>
1. 이전에 구현하고 사전 훈련한 GPT 모델을 수정해서 분류 미세 튜닝을 수행할 수 있다.

2. 분류 미세 튜닝에 유용한 예제로 '스펨'과 '스팸 아님'으로 구성된 텍스트 메시지 데이터셋을 사용한다.
   - 텍스트 메시지는 일반적으 이메일이 아니라 핸드폰으로 전달되지만, 동일한 단계가 이메일 분류에도 적용된다.

   ![프로세스](image/06-01-process2.png)

In [2]:
# 데이터셋 다운로드
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path}가 이미 있어 다운로드 및 압축 해제를 건너뜁니다.")
        return

    with urllib.request.urlopen(url) as response:
        with open(zip_path, 'wb') as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_path)
   
    # .tsv 파일 확장자를 추가합니다.
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"파일이 다운로드되어 {data_file_path}에 저장되었습니다.")
    
download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


파일이 다운로드되어 sms_spam_collection/SMSSpamCollection.tsv에 저장되었습니다.


In [4]:
# 판다스 데이터프레임으로 로드
import pandas as pd
df = pd.read_csv(data_file_path, sep='\t', header=None, names=['Label', 'Text'])
print(df)

     Label                                               Text
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham               Will ü b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name

[5572 rows x 2 columns]


In [5]:
# 레이블 분포 조사
#   - 레이블이 'ham'인 메시지와 'spam'인 메시지의 개수를 출력
print(df['Label'].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:
# 미세퓨닝을 빠르게 진행하기 위해 747개의 샘플만 포함되도록 데이터셋 줄이기
def create_balanced_dataset(df):

    # "스팸" 샘플 개수 세기
    num_spam = df[df["Label"] == "spam"].shape[0]

    # "스팸" 샘플 개수와 일치하도록 "햄" 샘플을 무작위로 샘플링
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)

    # "햄"과 "스팸"을 합침
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [ ]:
# 문자열로 된 클래스 레이블 "ham"과 "spam"을 정수로 변환
#   - 텍스트를 토큰 ID로 변환하는 것과 유사하다.
#   - 50,000개 단어 이상으로 구성된 GPT 어휘 사전을 사용하지 않고 0과 1로 레이블을 인코딩
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})
balanced_df

,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?
...,...,...
5537,1,Want explicit SEX in 30 secs? Ring 02073162414...
5540,1,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,1,Had your contract mobile 11 Mnths? Latest Moto...
5566,1,REMINDER FROM O2: To get 2.50 pounds free call...


In [13]:
# 데이터셋을 세 부분으로 분활하는 random_split 함수 정의
#   - 훈련 세트(70%), 검증 세트(10%), 테스트 세트(20%)
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)  # 데이터프레임을 무작위로 섞기
    train_end = int(len(df) * train_frac) # 훈련 세트 분할할 인덱스 계산
    validation_end = train_end + int(len(df) * validation_frac) # 검증 세트 분할할 인덱스 계산
    
    train_df = df[:train_end] # 훈련 세트
    validation_df = df[train_end:validation_end] # 검증 세트
    test_df = df[validation_end:] # 테스트 세트

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

In [ ]:
# 데이터셋을 나중에 사용하기 위해 CSV 파일로 저장
train_df.to_csv("train.csv", index=False)
validation_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)